# Week 8 Exercise — Agent with Tool Calling

**"The Price is Right"** — A small **planning agent** that can call **tools**: get the current time and estimate a product's price. Uses OpenAI-compatible **function calling** (same pattern as Week 8 Day 4) with **OpenRouter**.

No dependency on week8 `agents`, Modal, or Chroma. Set `OPENROUTER_API_KEY` in `.env`.

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
MODEL = "openai/gpt-4o-mini"

## Tools (Week 8 style)

Two tools: current time and product price estimate. The LLM decides when to call them.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time in ISO format",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_product_price",
            "description": "Estimate the retail price in USD of a product from its short description",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {"type": "string", "description": "Short product description (e.g. wireless earbuds 24hr battery)"}
                },
                "required": ["description"],
                "additionalProperties": False
            }
        }
    }
]

def get_current_time() -> str:
    return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")

def estimate_product_price(description: str) -> str:
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"Estimate the price in USD of this product. Reply with only the number.\n\n{description}"}],
            max_tokens=10
        )
        content = (r.choices[0].message.content or "").strip()
        return f"Estimated price: ${content}" if content else "Could not estimate."
    except Exception as e:
        return f"Error: {e}"

In [ ]:
SYSTEM = "You are a helpful assistant with access to tools: get_current_time and estimate_product_price. Use them when the user asks for the time or a price estimate."

def run_tool(name: str, arguments: dict) -> str:
    if name == "get_current_time":
        return get_current_time()
    if name == "estimate_product_price":
        return estimate_product_price(arguments.get("description", ""))
    return "Unknown tool"

def agent_chat(user_message: str, history: list) -> tuple:
    messages = [{"role": "system", "content": SYSTEM}]
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": user_message})
    max_rounds = 5
    while max_rounds > 0:
        max_rounds -= 1
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        msg = resp.choices[0].message
        if not getattr(msg, "tool_calls", None):
            return msg.content or "", history + [{"role": "user", "content": user_message}, {"role": "assistant", "content": msg.content or ""}]
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": msg.tool_calls})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments) if tc.function.arguments else {}
            result = run_tool(tc.function.name, args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "(Agent reached max rounds)", history + [{"role": "user", "content": user_message}, {"role": "assistant", "content": "(Max rounds)"}]

def chat_for_ui(message, history):
    reply, new_history = agent_chat(message, history)
    return reply

## Gradio UI

Chat with the agent. Try: "What time is it?" or "How much would wireless Bluetooth earbuds cost?" or both in one message.

In [ ]:
gr.ChatInterface(
    fn=chat_for_ui,
    type="messages",
    title="Week 8 — Agent with Tools",
    description="Ask for the current time or a product price estimate. The agent will call the right tools.",
    examples=["What time is it?", "Estimate the price of wireless Bluetooth earbuds."],
).launch()